In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
import sys
print(sys.version)

sys.path.append("/kaggle/input/alexnetmodel")
print("sys.path:", sys.path)
if os.path.exists("/kaggle/input/alexnetmodel"):
    print("Archivos en /kaggle/input/alexnetmodel:", os.listdir("/kaggle/input/alexnetmodel"))


In [ ]:
import tensorflow as tf
import tensorflow.keras as kt
from tensorflow.keras import layers, models, applications
import alexnet as alexnet
import time
import matplotlib.pyplot as plt
import tensorflow as tf
import sys
import numpy as np
import pandas as pd
from tensorflow.keras.optimizers import Adam, Nadam # type: ignore
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau # type: ignore
from tensorflow.keras.models import load_model
from sklearn.model_selection import train_test_split, KFold # type: ignore
from astropy.io import fits
import random
import configparser
plt.rc('axes', labelsize = 15)
plt.rc('xtick', labelsize = 10)
plt.rc('ytick', labelsize = 10)

# --- CONFIGURACIÓN ---
# Colores de la terminal
CYAN = '\033[36m'
YELLOW = '\033[33m'
GREEN = '\033[32m'
RED = '\033[31m'
ENDC = '\033[0m'

PRUEBA = 1
CLASSES = 4
MAIN_PATH = '/kaggle/working/'
RESULTS_PATH = os.path.join(MAIN_PATH, f'alexnet/')
TFRECORD_PATH_TRAIN = "/kaggle/input/train-dataset/train/"
#TFRECORD_PATH_TEST = 'drive/MyDrive/Data_set_lens (1)/tfrecords/train/' # Ojo: aquí tenías la misma ruta, verifica si es correcto
LEARNING_RATE = 1e-4
LABELS = ['theta_E', 'f_axis', 'e1', 'e2']
NUM_PIX = 100
CHANNLES = 1
DROPOUTS = (0.1, 0.5)
BATCH_SIZE = 512
EPOCHS = 50
INPUT_SHAPE = (NUM_PIX, NUM_PIX, CHANNLES)
dp1, dp2 = DROPOUTS
MODELS = ['alexnet_original','resnet','convnext','alexnet_new','alexnet_new_2']

# K-fold configuration
N_FOLDS = 5 # 5 iteraciones

# SEED GLOBAL
seed_value = 42
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)
tf.random.set_seed(seed_value)

print(f'Python version: {sys.version}')
print(f'Numpy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Tensorflow version: {tf.__version__}')
print(f'Keras version: {kt.__version__}')

In [ ]:
def get_weighted_loss(loss_weights=None, num_outputs=4):
    ''' Función de pérdida ponderada para múltiples salidas. '''
    if loss_weights is None:
        loss_weights = [1.0] * num_outputs
        if num_outputs >= 2:
            loss_weights[-2] = 3.0
            loss_weights[-1] = 3.0

    def weighted_mse(y_true, y_pred):
        total_loss = 0
        for i in range(num_outputs):
            true_val = y_true[:, i]
            pred_val = y_pred[:, i]
            mse = tf.reduce_mean(tf.square(true_val - pred_val))
            total_loss += loss_weights[i] * mse
        return total_loss
    return weighted_mse

def Plot_Metrics(history, metric, path, fold_idx):
    ''' Plots metrics guardando el indice del fold '''
    plt.figure()
    try:
        plt.plot(history.history[f'{metric}'], label = f'Training {metric}', c = 'k', lw = 0.8)
        plt.plot(history.history[f'val_{metric}'], label = f'Validation {metric}', c = 'r', lw = 0.8)
        plt.title(f'{metric.upper()} - FOLD {fold_idx}')
        plt.xlabel('epoch')
        plt.ylabel(metric)
        plt.legend()
        plt.savefig(path + f'{metric.lower()}_fold_{fold_idx}.png')
        plt.close()
    except KeyError:
        print(f"{RED}Metric {metric} not found in history{ENDC}")

def parse_tfrecord(example_proto):
    ''' Function to parse the TFRecord file. '''
    feature_description = {
        'image': tf.io.FixedLenFeature([], tf.string),
        'theta_E': tf.io.FixedLenFeature([], tf.float32),
        'f_axis': tf.io.FixedLenFeature([], tf.float32),
        'e1': tf.io.FixedLenFeature([], tf.float32),
        'e2': tf.io.FixedLenFeature([], tf.float32),
    }
    parsed_example = tf.io.parse_single_example(example_proto, feature_description)
    image = tf.io.decode_raw(parsed_example['image'], tf.float32)
    image = tf.reshape(image, INPUT_SHAPE)
    # Normalization
    image = (image - tf.reduce_min(image)) / (tf.reduce_max(image) + 1e-6)
    label = tf.stack([parsed_example[label] for label in LABELS], axis = 0)
    return image, label

def count_tfrecord_examples(tfrecord_files):
    total = 0
    for file in tfrecord_files:
        total += sum(1 for _ in tf.data.TFRecordDataset(file))
    return total

def get_models(model_name = 'alexnet_base', input_shape = (100,100,1), num_params=4, dropouts=(0.2,0.5)):
    match model_name:
    # 1 - Alexnet base
        case 'alexnet_original':
            alexnet_base = models.Sequential([
                layers.Input(shape=input_shape),
                layers.Conv2D(96, (11,11), strides=4, padding='valid', activation='relu'),
                layers.MaxPooling2D(pool_size=(3,3), strides=2),
                layers.Conv2D(256, (5,5), padding='same', activation='relu'),
                layers.MaxPooling2D(pool_size=(3,3), strides=2),
                layers.Conv2D(384, (3,3), padding='same', activation='relu'),
                layers.Conv2D(384, (3,3), padding='same', activation='relu'),
                layers.Conv2D(256, (3,3), padding='same', activation='relu'),
                layers.MaxPooling2D(pool_size = (3,3), strides=2),
                layers.Flatten(),
                layers.Dense(4096, activation='relu'),
                layers.Dropout(0.5),
                layers.Dense(4096, activation='relu'),
                layers.Dropout(0.5),
                layers.Dense(num_params, activation='linear')
            ], name="AlexNet_Base")
            model = alexnet_base

        case  'alexnet_new_2':
            alexnet_new_2 = models.Sequential([
                layers.Input(shape=input_shape),
                layers.Conv2D(96, (11,11), strides=4, padding='valid', activation='relu'),
                layers.MaxPooling2D(pool_size=(3,3), strides=2),
                layers.Conv2D(256, (5,5), padding='same', activation='relu'),
                layers.MaxPooling2D(pool_size=(3,3), strides=2),
                layers.Conv2D(384, (3,3), padding='same', activation='relu'),
                layers.Conv2D(384, (3,3), padding='same', activation='relu'),
                layers.Conv2D(256, (3,3), padding='same', activation='relu'),
                layers.Conv2D(128, (3,3), padding='same', activation='relu'),
                layers.MaxPooling2D(pool_size = (3,3), strides=2),
                layers.Flatten(),
                layers.Dense(4096, activation='relu'),
                layers.Dropout(0.5),
                layers.Dense(4096, activation='relu'),
                layers.Dropout(0.5),
                layers.Dense(num_params, activation='linear')
            ], name="AlexNet_Base")
            model = alexnet_new_2
        
        case 'resnet':
            # 2. ResNet50V2
            # Nota: ResNet espera 3 canales, repetiremos el canal de grises.
            base_resnet = applications.ResNet50V2(include_top=False, weights=None, input_shape=input_shape, pooling='avg')
            model = models.Sequential([base_resnet, layers.Dense(num_params, activation='linear')], name="ResNet50")

        case 'convnext':
            # 3. ConvNeXt Tiny
            base_convnext = applications.ConvNeXtTiny(include_top=False, weights=None, input_shape=input_shape, pooling='avg')
            model = models.Sequential([base_convnext, layers.Dense(num_params, activation='linear')], name="ConvNeXt")

        case 'alexnet_new':
            # 4. Tu modelo propio (Ejemplo de estructura basada en AlexNet)
            # Aquí deberías insertar la definición exacta de tu arquitectura.
            dp1, dp2 = dropouts
            model = alexnet.AlexNet(input_shape=INPUT_SHAPE, classes=CLASSES, dp1=dp1, dp2=dp2)

    return model

In [ ]:
def main():
    # Lista de modelos a entrenar
    import gc
    try:
        print(f'{YELLOW}Iniciando Pipeline Multi-Modelo K-Fold{ENDC}\n')
        train_tfrecord_files = sorted([os.path.join(TFRECORD_PATH_TRAIN, f) for f in os.listdir(TFRECORD_PATH_TRAIN) if f.endswith('.tfrecord')])
        num_train_total = count_tfrecord_examples(train_tfrecord_files)
        num_files = len(train_tfrecord_files)
        full_ds = tf.data.TFRecordDataset(train_tfrecord_files).map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)

        model_name = MODELS[3]
    
        print(f"\n{GREEN}########################################")
        print(f"TRAINING ARCHITECTURE: {model_name.upper()}")
        print(f"########################################{ENDC}")
        
        # Crear subcarpeta para cada arquitectura
        MODEL_RESULTS_PATH = os.path.join(RESULTS_PATH, model_name)
        os.makedirs(MODEL_RESULTS_PATH, exist_ok=True)
        for fold in range(N_FOLDS):
            print(f"\n{CYAN}--- {model_name} | Fold {fold + 1}/{N_FOLDS} ---{ENDC}")
            
            files_per_fold = num_files // N_FOLDS
            start_idx = fold * files_per_fold
            end_idx = (fold + 1) * files_per_fold if fold != N_FOLDS - 1 else num_files
            
            val_files = train_tfrecord_files[start_idx:end_idx]
            train_files = [f for f in train_tfrecord_files if f not in val_files]
            
            # Creamos los datasets solo con los archivos que corresponden
            train_dataset_raw = tf.data.TFRecordDataset(train_files).map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
            val_dataset_raw = tf.data.TFRecordDataset(val_files).map(parse_tfrecord)
            
            tf.keras.backend.clear_session()
            gc.collect() 

            model = get_models(model_name=model_name, dropouts = DROPOUTS)
            
            # Cálculo de pasos basado en los archivos reales
            # (Asumiendo que conoces cuántos ejemplos hay por archivo o el total)
            num_val_fold = count_tfrecord_examples(val_files)
            num_train_fold = count_tfrecord_examples(train_files)
            
            steps_per_epoch = num_train_fold // BATCH_SIZE
            validation_steps = num_val_fold // BATCH_SIZE

            # Pipeline final
            train_dataset = (train_dataset_raw.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE).repeat())
            val_dataset = (val_dataset_raw.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
            
            optimizer = Nadam(learning_rate=LEARNING_RATE)
            loss_fn = get_weighted_loss([1.0, 1.0, 3.0, 3.0], num_outputs=CLASSES)
            
            model.compile(optimizer=optimizer, loss = loss_fn, metrics=['mae'])

            reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr = 1e-8, verbose=0)
            #early_stop = EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)

            # Entrenamiento
            start = time.time()
            history = model.fit(
                train_dataset,
                validation_data=val_dataset,
                epochs=EPOCHS,
                steps_per_epoch=steps_per_epoch,
                validation_steps=validation_steps,
                callbacks=[reduce_lr],
                verbose=1
            )
            end = time.time()

            prefix = f"{model_name}_fold_{fold+1}"
            
            # Guardar modelo (.keras es el formato recomendado actualmente)
            model.save(os.path.join(MODEL_RESULTS_PATH, f'{prefix}.keras'))
            
            # Guardar historial y gráficas
            pd.DataFrame(history.history).to_csv(os.path.join(MODEL_RESULTS_PATH, f'{prefix}_history.csv'))
            Plot_Metrics(history, 'mae', MODEL_RESULTS_PATH + "/", fold+1)
            Plot_Metrics(history, 'loss', MODEL_RESULTS_PATH + "/", fold+1)
            
            print(f"{GREEN}Terminado {prefix} en {(end-start)/60:.2f} min{ENDC}")
            del model
            gc.collect()

    except Exception as e:
        print(f'{RED}Error in line {sys.exc_info()[2].tb_lineno}: {e}{ENDC}')

if __name__ == '__main__':
    main()